# 22장 Transformer 직접 구현 — 어텐션을 쌓는다

**1단계 · 따라 하기** — 『법률가를 위한 인공지능 코딩 자습서』 실습 노트북

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch22_1_follow.ipynb)

이 노트북에는 책의 코드가 순서대로 들어 있습니다. **위에서부터 셀을 하나씩 실행**하고(셀을 누른 뒤 `Shift+Enter`), 출력이 책과 같은지 대조하세요.

- 셀 아래에 미리 보이는 출력은 저자의 컴퓨터(CPU)에서 실행한 결과입니다. 걸린 시간은 환경마다 다릅니다.
- 책은 코드를 **붙여 넣지 말고 직접 치라**고 권합니다. 실행해 본 다음, 셀 아래에 빈 코드 셀을 만들어(`+ 코드`) 같은 코드를 눈으로 읽으며 쳐 보세요.
- 설명은 책에 있습니다. 이 노트북은 실행을 위한 것입니다.

| 단계 | 할 일 | 노트북 |
|---|---|---|
| 1. 따라 하기 | `transformer.py` 와 `p01_compare.py` 를 치고 실행. `MultiHeadAttention.forward` 의 shape을 종이에 먼저 | [ch22_1_follow.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch22_1_follow.ipynb) **(지금 이 노트북)** |
| 2. 주석 달기 | `forward` 각 줄 뒤에 shape을, `EncoderBlock` 두 잔차 줄에 17장 대응을 주석으로 | [ch22_2_comment.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch22_2_comment.ipynb) |
| 3. 다시 만들기 | 주석만 남긴 파일에서 `MultiHeadAttention` 복원(가장 어려운 복원 과제). 그다음 14장 패키지에 `--model transformer` 등록 | [ch22_3_build.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch22_3_build.ipynb) |

In [ ]:
# 준비 셀 — 맨 먼저 한 번 실행하세요. 실습 코드와 데이터를 받아 옵니다.
import os, sys
if "google.colab" in sys.modules:                       # Colab 에서 열었을 때
    ROOT = "/content/AI-Coding-for-Lawyers"
    if not os.path.isdir(ROOT):
        !git clone -q --depth 1 https://github.com/BowonKwon/AI-Coding-for-Lawyers.git {ROOT}
else:                                                   # 내 컴퓨터에서 열었을 때 (저장소 폴더 안)
    ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(ROOT, "notebooks")) and os.path.dirname(ROOT) != ROOT:
        ROOT = os.path.dirname(ROOT)
os.environ.setdefault("JUCOS_DATA", os.path.join(ROOT, "data"))   # 데이터를 두는 폴더
os.chdir(os.path.join(ROOT, "code", "22"))                      # 이 장의 코드 폴더로 이동
print("준비 끝. 작업 폴더: code/22")

준비 끝. 작업 폴더: code/22


## `transformer.py` — 22장 Transformer 인코더를 직접 만든다: 위치 인코딩, 멀티헤드 어텐션, 인코더 블록, 분류기

이 파일은 준비 셀에서 이미 받아 두었습니다. 내용은 아래와 같습니다. (왼쪽 폴더 모양 아이콘에서 파일을 두 번 누르면 열어서 고칠 수도 있습니다.)

```python
# 22장 Transformer 인코더를 직접 만든다: 위치 인코딩, 멀티헤드 어텐션, 인코더 블록, 분류기
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class PositionalEncoding(nn.Module):
    """토큰 순서 정보를 더해 준다. 어텐션 자체는 순서를 모르기 때문."""

    def __init__(self, d_model, max_len=512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()                    # [L, 1]
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)                                  # 짝수 차원: sin
        pe[:, 1::2] = torch.cos(pos * div)                                  # 홀수 차원: cos
        self.register_buffer("pe", pe)                                      # 파라미터 아님(학습 안 함), 저장은 됨

    def forward(self, x):                                                   # x: [B, L, d]
        return x + self.pe[: x.size(1)]


class MultiHeadAttention(nn.Module):
    """헤드 h 개가 각자 다른 관점으로 어텐션을 계산하고 결과를 이어 붙인다."""

    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.h, self.dk = n_heads, d_model // n_heads
        self.wq = nn.Linear(d_model, d_model)                                # 질의·키·값을 만드는 선형층
        self.wk = nn.Linear(d_model, d_model)
        self.wv = nn.Linear(d_model, d_model)
        self.wo = nn.Linear(d_model, d_model)                                # 헤드를 합친 뒤 한 번 더

    def forward(self, x, pad_mask=None):                                    # x: [B, L, d], pad_mask: [B, L] True=패딩
        B, L, d = x.shape
        # [B, L, d] → [B, h, L, dk] : 헤드마다 dk 차원씩 나눠 본다
        q = self.wq(x).view(B, L, self.h, self.dk).transpose(1, 2)
        k = self.wk(x).view(B, L, self.h, self.dk).transpose(1, 2)
        v = self.wv(x).view(B, L, self.h, self.dk).transpose(1, 2)
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.dk)               # [B, h, L, L]  (21장의 x @ x.T)
        if pad_mask is not None:
            scores = scores.masked_fill(pad_mask[:, None, None, :], float("-inf"))   # 패딩은 보지 않는다
        attn = F.softmax(scores, dim=-1)                                    # 행 합 = 1
        out = attn @ v                                                      # [B, h, L, dk]
        out = out.transpose(1, 2).reshape(B, L, d)                          # 헤드를 다시 이어 붙임
        return self.wo(out), attn


class EncoderBlock(nn.Module):
    """어텐션 → 잔차+정규화 → 피드포워드 → 잔차+정규화. 17장 ResNet 의 지름길이 여기에도 있다."""

    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, pad_mask=None):
        a, attn = self.attn(self.norm1(x), pad_mask)
        x = x + self.drop(a)                                                # 잔차 연결 1
        x = x + self.drop(self.ff(self.norm2(x)))                           # 잔차 연결 2
        return x, attn


class TransformerClassifier(nn.Module):
    """임베딩 + 위치 → 인코더 블록 n 개 → 패딩 제외 평균 → 분류"""

    def __init__(self, vocab_size, num_classes, d_model=64, n_heads=4, d_ff=128, n_layers=2, max_len=64):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos = PositionalEncoding(d_model, max_len)
        self.blocks = nn.ModuleList([EncoderBlock(d_model, n_heads, d_ff) for _ in range(n_layers)])
        self.norm = nn.LayerNorm(d_model)
        self.fc = nn.Linear(d_model, num_classes)

    def forward(self, ids, return_attn=False):                              # ids: [B, L]
        pad_mask = ids == 0
        x = self.pos(self.emb(ids) * math.sqrt(self.emb.embedding_dim))
        attns = []
        for blk in self.blocks:
            x, attn = blk(x, pad_mask)
            attns.append(attn)
        x = self.norm(x)
        keep = (~pad_mask).unsqueeze(-1).float()
        sent = (x * keep).sum(1) / keep.sum(1).clamp(min=1)                 # 19장과 같은 평균 — 다만 문맥이 섞인 뒤
        out = self.fc(sent)
        return (out, attns) if return_attn else out


if __name__ == "__main__":
    torch.manual_seed(0)
    ids = torch.tensor([[5, 3, 9, 0, 0], [7, 2, 4, 8, 1]])
    m = TransformerClassifier(vocab_size=20, num_classes=3)
    out, attns = m(ids, return_attn=True)
    print("파라미터:", f"{sum(p.numel() for p in m.parameters()):,}")
    print("출력:", tuple(out.shape), "| 블록 수", len(attns), "| 어텐션 shape [B, h, L, L]:", tuple(attns[0].shape))
    print("첫 문장 1층 헤드0 어텐션 (패딩 열은 0):")
    for row in attns[0][0, 0].tolist():
        print("  ", [round(v, 2) for v in row])
    pe = PositionalEncoding(8, 6).pe
    print("위치 인코딩 [6, 8] 앞 3행:")
    for row in pe[:3].tolist():
        print("  ", [round(v, 2) for v in row])
```

In [ ]:
!python transformer.py

파라미터: 68,547
출력: (2, 3) | 블록 수 2 | 어텐션 shape [B, h, L, L]: (2, 4, 5, 5)
첫 문장 1층 헤드0 어텐션 (패딩 열은 0):
   [0.5, 0.32, 0.19, 0.0, 0.0]
   [0.4, 0.42, 0.18, 0.0, 0.0]
   [0.24, 0.33, 0.44, 0.0, 0.0]
   [0.43, 0.39, 0.17, 0.0, 0.0]
   [0.38, 0.41, 0.22, 0.0, 0.0]
위치 인코딩 [6, 8] 앞 3행:
   [0.0, 1.0, 0.0, 1.0, 0.0, 1.0, 0.0, 1.0]
   [0.84, 0.54, 0.1, 1.0, 0.01, 1.0, 0.0, 1.0]
   [0.91, -0.42, 0.2, 0.98, 0.02, 1.0, 0.0, 1.0]


## `p01_compare.py` — 22.5 같은 데이터에서 단어 주머니 · BiLSTM · Transformer 비교, 그리고 어텐션이 무엇을 보는지

이 파일은 준비 셀에서 이미 받아 두었습니다. 내용은 아래와 같습니다. (왼쪽 폴더 모양 아이콘에서 파일을 두 번 누르면 열어서 고칠 수도 있습니다.)

```python
# 22.5 같은 데이터에서 단어 주머니 · BiLSTM · Transformer 비교, 그리고 어텐션이 무엇을 보는지
import os
import sys
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "19"))
sys.path.insert(0, os.path.join(os.path.dirname(__file__), "..", "20"))
from statutes_dataset import StatuteDataset
from rnn_models import TorchRNNClassifier
from transformer import TransformerClassifier

LAWS = ["민법", "형법", "민사소송법", "형사소송법"]
torch.manual_seed(0)
train_set = StatuteDataset(LAWS, "train")
test_set = StatuteDataset(LAWS, "test", vocab=train_set.vocab)
test_loader = DataLoader(test_set, batch_size=256)
V, C = len(train_set.vocab), len(LAWS)


class BagOfWords(nn.Module):
    def __init__(self, vocab_size, dim, num_classes):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, dim, padding_idx=0)
        self.fc = nn.Linear(dim, num_classes)

    def forward(self, ids):
        vec = self.emb(ids)
        mask = (ids != 0).unsqueeze(-1).float()
        return self.fc((vec * mask).sum(1) / mask.sum(1).clamp(min=1))


def accuracy(model):
    model.eval()
    c = 0
    with torch.no_grad():
        for x, y in test_loader:
            c += (model(x).argmax(1) == y).sum().item()
    model.train()
    return c / len(test_set)


def run(name, model, epochs=8, lr=1e-3):
    torch.manual_seed(0)
    loader = DataLoader(train_set, batch_size=32, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    t0 = time.time()
    best = 0.0
    for _ in range(epochs):
        for x, y in loader:
            opt.zero_grad()
            loss_fn(model(x), y).backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
        best = max(best, accuracy(model))
    n = sum(p.numel() for p in model.parameters())
    print(f"{name:<12} 파라미터 {n:>9,}  최고 정확도 {best:.3f}  마지막 {accuracy(model):.3f}  ({time.time() - t0:.0f}초)")
    return model


if __name__ == "__main__":
    print(f"학습 {len(train_set)} / 시험 {len(test_set)} / 어휘 {V}")
    run("단어 주머니", BagOfWords(V, 64, C), lr=2e-3)
    run("BiLSTM", TorchRNNClassifier(V, 64, 64, C, "lstm", bidirectional=True), lr=2e-3)
    tf = run("Transformer", TransformerClassifier(V, C, d_model=64, n_heads=4, d_ff=128, n_layers=2, max_len=64))

    # 학습된 Transformer 가 어느 토큰을 보는가: 시험 조문 하나의 마지막 층 어텐션(헤드 평균), 각 토큰이 가장 많이 본 토큰
    tf.eval()
    x, y = test_set[3]
    text = test_set.items[3][0]
    toks = [train_set.vocab.itos[i] for i in x.tolist() if i != 0]
    with torch.no_grad():
        out, attns = tf(x.unsqueeze(0), return_attn=True)
    a = attns[-1][0].mean(0)[: len(toks), : len(toks)]                     # [L, L]
    print(f"\n조문: {text[:60]}…  (정답 {LAWS[y]}, 예측 {LAWS[out.argmax().item()]})")
    print("각 토큰이 가장 크게 본 토큰 (마지막 층, 헤드 평균):")
    for i, t in enumerate(toks[:10]):
        j = a[i].argmax().item()
        print(f"  {t:<12} → {toks[j]:<12} ({a[i, j]:.2f})")
```

In [ ]:
!python p01_compare.py

학습 2108 / 시험 526 / 어휘 5986
단어 주머니       파라미터   383,364  최고 정확도 0.892  마지막 0.892  (2초)
BiLSTM       파라미터   450,180  최고 정확도 0.838  마지막 0.816  (20초)
Transformer  파라미터   450,436  최고 정확도 0.838  마지막 0.838  (13초)

조문: 제163조(당사자의 참여권, 신문권) ①검사, 피고인 또는 변호인은 증인신문에 참여할 수 있다. ②증인신문의…  (정답 형사소송법, 예측 형사소송법)
각 토큰이 가장 크게 본 토큰 (마지막 층, 헤드 평균):
  <unk>        → 예외로          (0.31)
  <unk>        → 예외로          (0.31)
  <unk>        → 예외로          (0.31)
  ①검사,         → 예외로          (0.27)
  피고인          → 변호인은         (0.22)
  또는           → 예외로          (0.24)
  변호인은         → 변호인은         (0.13)
  증인신문에        → 예외로          (0.15)
  참여할          → 예외로          (0.15)
  수            → 예외로          (0.19)


## 다음 단계

- 2단계 「주석 달기」: [ch22_2_comment.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch22_2_comment.ipynb)
- 3단계 「다시 만들기」: [ch22_3_build.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch22_3_build.ipynb)
- 책의 「확인 문제」는 이 노트북 끝에 셀을 추가해 풀어 보세요.